
# Lab 1: Program Integrity — Improper Payment Flags (Gold)

Reducing **improper payments** is one of the largest program-integrity missions for
a state Medicaid agency. CMS reports Medicaid improper payments in the tens of
billions of dollars a year, and TMSIS is the data agencies use to find them.

In this notebook we build two gold tables. The first applies a set of **claim edit checks**
(deterministic rules) to the silver TMSIS claims and flags claims that look like
improper payments or potential fraud. Nothing in the data marks a claim as
fraudulent — we find them by writing the rules.

## Objectives
1. **Apply edit checks**: encode common Medicaid improper-payment rules in SQL.
2. **Create a flags table**: one row per (claim, rule) in the gold layer.
3. **Create a provider summary table**: aggregate flagged claims and dollars by provider.

Run the table-building cells below. You will explore the results later in
**Lab 4 (Data Warehouse)** and **Lab 5 (Asking Natural Language Questions)**.

## Rules applied
| Rule | What it catches |
|------|-----------------|
| `DUPLICATE_CLAIM` | Same beneficiary, provider, service date, and procedure billed more than once |
| `PAID_BEFORE_SERVICE` | Payment date precedes the service begin date |
| `DISCHARGE_BEFORE_ADMISSION` | Discharge date precedes the admission date |
| `SERVICE_BEFORE_BIRTH` | Service date precedes the beneficiary's birth date |
| `OUTPATIENT_OVERPAYMENT` | Outpatient office visit paid far above the normal range |
| `BENE_MULTI_STATE_SAME_DAY` | One beneficiary billed in two states on the same day |
| `EXCESSIVE_DAILY_VOLUME` | Implausible number of visits for one beneficiary in a single day |

Let's get started!

####Setup
First step in all the labs will be to run a setup notebook. This notebook to create a catalog to store data and to set your default catalog

In [0]:
%run "../Lab 0 - Setup/SetCatalogName"

### Build the flags table
Each rule below produces the same set of columns so we can `UNION ALL` them into a
single gold table. A claim that trips more than one rule appears once per rule.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.tmsis_improper_payment_flags AS

-- Rule 1: Duplicate paid claims
WITH duplicates AS (
  SELECT *,
    COUNT(*) OVER (
      PARTITION BY BENE_ID, PRVDR_ID, SRVC_BGNNG_DT, PRCDR_CD_1
    ) AS claim_count
  FROM silver.tmsis_claims
),
-- Rule 6: Same beneficiary billed in more than one state on the same day.
-- Spark SQL does not allow COUNT(DISTINCT ...) as a window function, so we
-- collect the distinct states into a set over the window and measure its size.
multi_state AS (
  SELECT *,
    SIZE(COLLECT_SET(SUBMTG_STATE_CD) OVER (
      PARTITION BY BENE_ID, SRVC_BGNNG_DT
    )) AS state_count
  FROM silver.tmsis_claims
),
-- Rule 7: Implausible number of visits for one beneficiary in a single day
daily_volume AS (
  SELECT *,
    COUNT(*) OVER (
      PARTITION BY BENE_ID, DATE_TRUNC('day', SRVC_BGNNG_DT)
    ) AS daily_claim_count
  FROM silver.tmsis_claims
)

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'DUPLICATE_CLAIM' AS flag_rule,
       'Same beneficiary, provider, service date, and procedure billed more than once' AS flag_description
FROM duplicates
WHERE claim_count > 1

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'PAID_BEFORE_SERVICE',
       'Medicaid payment date precedes the service begin date'
FROM silver.tmsis_claims
WHERE MDCD_PD_DT < SRVC_BGNNG_DT

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'DISCHARGE_BEFORE_ADMISSION',
       'Discharge date precedes the admission date'
FROM silver.tmsis_claims
WHERE DSCHRG_DT < ADMISSION_DT

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'SERVICE_BEFORE_BIRTH',
       'Service date precedes the beneficiary birth date'
FROM silver.tmsis_claims
WHERE SRVC_BGNNG_DT < BIRTH_DT

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'OUTPATIENT_OVERPAYMENT',
       'Outpatient claim paid far above the normal outpatient range'
FROM silver.tmsis_claims
WHERE BILL_TYPE_CD = '131' AND TOT_MDCD_PD_AMT > 1000

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'BENE_MULTI_STATE_SAME_DAY',
       'Beneficiary billed in more than one state on the same day'
FROM multi_state
WHERE state_count > 1

UNION ALL

SELECT ICN_NUM, BENE_ID, PRVDR_ID, SUBMTG_STATE_CD, SRVC_BGNNG_DT, TOT_MDCD_PD_AMT,
       'EXCESSIVE_DAILY_VOLUME',
       'Implausible number of claims for one beneficiary in a single day'
FROM daily_volume
WHERE daily_claim_count >= 15

### Build the provider risk summary table
Create a gold table that aggregates flagged claims, distinct rules, and flagged
dollars for each provider. You will examine the provider results in a later lab.


In [0]:
%sql
CREATE OR REPLACE TABLE gold.tmsis_provider_risk_summary AS
SELECT PRVDR_ID,
       COUNT(*) AS flagged_claims,
       COUNT(DISTINCT flag_rule) AS distinct_rules,
       ROUND(SUM(TOT_MDCD_PD_AMT), 0) AS flagged_dollars
FROM gold.tmsis_improper_payment_flags
GROUP BY PRVDR_ID
ORDER BY flagged_dollars DESC;


### Tables ready for the later labs
You have created two gold tables:

* `gold.tmsis_improper_payment_flags` — one row per flagged claim and rule
* `gold.tmsis_provider_risk_summary` — flagged claims and dollars aggregated by provider

You will look at the results in **Lab 4 (Data Warehouse)** and
**Lab 5 (Asking Natural Language Questions)**. There you will explore the flags,
compare providers, and ask questions such as "Which providers have the most
duplicate claims?" and "Show me claims paid before the service date."
